In [1]:
# general
import numpy as np
import pandas as pd
import geopandas as gpd
import sys
import gc
import glob
import swifter
from h3 import h3
import requests
import zipfile
import io
import json
# import pygrts

# import shapely.geometry as sg
from shapely import Point, Polygon, buffer

# from sklearn.cluster import DBSCAN

# plotting and output
from tqdm.auto import tqdm
# import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings(action='ignore')
tqdm.pandas()

/home/cbutsko/miniconda3/envs/py310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [23]:
label_columns = ['cropland', 'landcover', 'cropgroup', 'croptype', 'sampling_croptype']
wc2ec_map = pd.read_csv('resources/wc2eurocrops_map.csv')
ec_map = pd.read_csv('resources/eurocrops_map_sampling_edition.csv', index_col='ec_code')

In [2]:
local_path = '/home/cbutsko/Desktop/cbutsko_experiments/tmp/'

tdir = '/home/cbutsko/Desktop/cbutsko_experiments/rdm_croptype_stats/'
files_lst = glob.glob('{}*items'.format(tdir))
files_lst = [xx for xx in files_lst if int(xx.split('/')[-1][:4])>2017]

In [53]:
files_lst[-20:]

['/home/cbutsko/Desktop/cbutsko_experiments/rdm_croptype_stats/2021kencopernicusgeoglamlrpoly111items',
 '/home/cbutsko/Desktop/cbutsko_experiments/rdm_croptype_stats/2021delseurocroppoly110items',
 '/home/cbutsko/Desktop/cbutsko_experiments/rdm_croptype_stats/2021kencopernicusgeoglamsrpoly111items',
 '/home/cbutsko/Desktop/cbutsko_experiments/rdm_croptype_stats/2021lkawapor1poly111items',
 '/home/cbutsko/Desktop/cbutsko_experiments/rdm_croptype_stats/2021lkawapor2poly111items',
 '/home/cbutsko/Desktop/cbutsko_experiments/rdm_croptype_stats/2021lteurocroppoly110items',
 '/home/cbutsko/Desktop/cbutsko_experiments/rdm_croptype_stats/2021lvfullpoly110items',
 '/home/cbutsko/Desktop/cbutsko_experiments/rdm_croptype_stats/2021mozwapor1poly111items',
 '/home/cbutsko/Desktop/cbutsko_experiments/rdm_croptype_stats/2021pteurocroppoly110items',
 '/home/cbutsko/Desktop/cbutsko_experiments/rdm_croptype_stats/2021rwawaporakagerapoly111items',
 '/home/cbutsko/Desktop/cbutsko_experiments/rdm_croptype

In [68]:
with open(files_lst[-3]) as f:
    tdata = json.load(f)

turl = [xx['value'] for xx in tdata if xx['name']=='CollectionDownloadUrl'][0]

In [69]:
r = requests.get(turl)
tzip = zipfile.ZipFile(io.BytesIO(r.content))
tzip.extractall(path=local_path)

In [70]:
shape_fnames = [xx for xx in glob.glob('{}*'.format(local_path)) if xx.endswith('.shp')]
gpkg_fnames = [xx for xx in glob.glob('{}*'.format(local_path)) if xx.endswith('.gpkg')]

In [71]:
gpkg_fnames

['/home/cbutsko/Desktop/cbutsko_experiments/tmp/france_centroids_trimmed.gpkg',
 '/home/cbutsko/Desktop/cbutsko_experiments/tmp/2018_AT_LPIS_centroids.gpkg']

In [72]:
shape_fnames

['/home/cbutsko/Desktop/cbutsko_experiments/tmp/2019_BE_Flanders_full_POLY_110.shp',
 '/home/cbutsko/Desktop/cbutsko_experiments/tmp/2020_CAN_AAFC-ACIGTD_POINT_110.shp',
 '/home/cbutsko/Desktop/cbutsko_experiments/tmp/2019_US_USDA2019cdls_POINT_110.shp',
 '/home/cbutsko/Desktop/cbutsko_experiments/tmp/2020_BR_LEM-FEB_POLY_110.shp',
 '/home/cbutsko/Desktop/cbutsko_experiments/tmp/2019_TZ_AFSIS_POINT_110.shp',
 '/home/cbutsko/Desktop/cbutsko_experiments/tmp/2020_RW_WAPOR_1_POINT_111.shp',
 '/home/cbutsko/Desktop/cbutsko_experiments/tmp/2019_LV_full_POLY_110.shp',
 '/home/cbutsko/Desktop/cbutsko_experiments/tmp/2018_EU_LUCAS2018_POINT_110.shp',
 '/home/cbutsko/Desktop/cbutsko_experiments/tmp/2021_UGA_COPERNICUS-GEOGLAM-SR_POLY_111.shp',
 '/home/cbutsko/Desktop/cbutsko_experiments/tmp/2021_UGA_COPERNICUS-GEOGLAM-LR_POLY_111.shp']

In [73]:
# shape_fname = gpkg_fnames[1]
shape_fname = shape_fnames[-1]

original_data = gpd.read_file(shape_fname)
ref_id = '_'.join(shape_fname.split('/')[-1].split('_')[:3])
# ref_id = '2020_FR_LPIS'

In [74]:
ref_id

'2021_UGA_COPERNICUS-GEOGLAM-LR'

In [75]:
original_data.shape

(7582, 31)

In [76]:
original_data['centroid'] = original_data['geometry'].centroid
original_data['lat'] = original_data['centroid'].swifter.apply(lambda xx: xx.y, axis=1)
original_data['lon'] = original_data['centroid'].swifter.apply(lambda xx: xx.x, axis=1)

original_data['h3_cell_res_3'] = original_data.swifter.apply(lambda xx: h3.geo_to_h3(
    xx['lat'],
    xx['lon'],
    resolution=3
    ), axis=1)
h3_cells_lst = original_data['h3_cell_res_3'].unique()

Pandas Apply: 100%|██████████| 7582/7582 [00:00<00:00, 91157.26it/s]


In [77]:
original_data['h3_cell_res_3'].nunique()

9

In [78]:
original_data['CT'] = original_data['CT'].astype(int)
original_data['LC'] = original_data['LC'].astype(int) 

original_data['CT'].replace(0, np.nan, inplace=True)
original_data['CT'].fillna(original_data['LC'], inplace=True)
original_data['ec_code'] = original_data['CT'].map(wc2ec_map.set_index('croptype')['ec_code'])

for tlevel in label_columns:
    original_data[tlevel] = original_data['ec_code'].map(ec_map['{}_label'.format(tlevel)]).astype('int32')
    original_data['{}_name'.format(tlevel)] = original_data['ec_code'].map(ec_map['{}_name'.format(tlevel)])

In [79]:
grid_samples = []
crops_to_sample = list(original_data['sampling_croptype_name'].unique())
crops_to_sample = [xx for xx in crops_to_sample if xx not in ['cropland_unspecified','unknown']]
original_data['is_grid_sample'] = False

pbar = tqdm(total=len(h3_cells_lst))
for h3_cell in h3_cells_lst:
    pbar.update(1)
    for crop in crops_to_sample:

        h3_all_samples = original_data[
            (original_data['h3_cell_res_3']==h3_cell) & 
            (original_data['sampling_croptype_name']==crop)
            ]
        
        if len(h3_all_samples)==0:
            continue

        if len(h3_all_samples)<=max_samples_per_cell:
            grid_samples.extend(list(h3_all_samples['sampleID'].values))
            continue
        
        h3_cell_buffered = buffer(Polygon(h3.h3_to_geo_boundary(h3_cell)), distance=-0.1)

        coords_grid = np.mgrid[
                    (np.min(h3_cell_buffered.envelope.exterior.coords.xy[1])+0.01):(np.max(h3_cell_buffered.envelope.exterior.coords.xy[1])-0.01):complex(0,np.ceil(np.sqrt(max_samples_per_cell))), 
                    (np.min(h3_cell_buffered.envelope.exterior.coords.xy[0])+0.01):(np.max(h3_cell_buffered.envelope.exterior.coords.xy[0])-0.01):complex(0,np.ceil(np.sqrt(max_samples_per_cell)))].reshape(2,-1).T
        coords_grid = coords_grid[[Polygon(h3.h3_to_geo_boundary(h3_cell)).contains(Point(xx[::-1])) for xx in coords_grid]]

        _grid_samples = []
        for grid_point in coords_grid:
            tsample = ((h3_all_samples['lon'] - grid_point[0]).abs() + (h3_all_samples['lat'] - grid_point[1]).abs()).argmin()
            _grid_samples.append(h3_all_samples.iloc[tsample]['sampleID'])
        grid_samples.extend(_grid_samples)

pbar.reset()

original_data['is_grid_sample'] = original_data['sampleID'].isin(grid_samples)
original_data[(original_data['is_grid_sample'])][['sampleID','lat','lon','sampling_croptype_name']].to_csv('/home/cbutsko/Desktop/cbutsko_experiments/{}_rdm_grid_samples.csv'.format(ref_id))

  0%|          | 0/8 [01:16<?, ?it/s]


In [80]:
tdata = original_data[(original_data['is_grid_sample'])][['sampleID','centroid','sampling_croptype_name','h3_cell_res_3']]
tdata = gpd.GeoDataFrame(tdata, geometry='centroid')
tdata.to_file('/home/cbutsko/Desktop/cbutsko_experiments/{}_rdm_grid_samples.gpkg'.format(ref_id), driver='GPKG')

In [82]:
tdata['sampling_croptype_name'].value_counts()

sampling_croptype_name
maize                         229
herbaceous_vegetation         219
root_tuber_crops              168
shrubland                     162
other_oilseeds                146
millet_sorghum                143
built_up                      136
dry_pulses_legumes            104
trees_unspecified              75
fruit_nuts                     50
vegetables_fruits              34
rice                           29
sunflower                      23
herb_spice_medicinal_crops     22
fibre_crops                    16
grass_fodder_crops              9
other_permanent_crops           9
soy_soybeans                    5
oats                            1
Name: count, dtype: int64